In [ ]:
"""
LEGACY: Yahoo Finance minute-data ingestion (kept as-is, not used by the IBKR ingest)

Status on 2026-10-06: not maintained. It needs pyspark and yfinance, which are NOT in requirements/venv_ingest_requirements.txt
(install them separately if you run it), it writes to the old /Users/nico/Desktop/IMGM_data/ folders, and it converts times
with a Qatar offset. Only its local_file_management import was updated (the old function names no longer exist; they are
aliased to the renamed functions of so.core.local_file_management).
"""
None

In [1]:
# TOOLS LIBRARIES
import pandas as pd
import json
import pyspark
from pyspark.sql import SparkSession
from datetime import datetime, date, timedelta
import time
import pytz
import plotly.express as px
import os
# IMPORT SPARK FUNCTIONS
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, LongType, StringType, IntegerType, DoubleType, FloatType
from pyspark.sql.functions import col, to_timestamp, to_date, lit, count, countDistinct, floor, unix_timestamp, avg, array, concat, collect_list, explode, sequence, struct, row_number, greatest, least
from pyspark.sql.window import Window
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql.connect.dataframe import DataFrame as ConnectSparkDataFrame
# DATA INGESTION LIBRARIES
import yfinance as yf
# FILE MANIMULATION
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS OF THE RESEARCH WORKSPACE (THE OLD NAMES NO LONGER EXIST)
from so.core.local_file_management import read_csv_file_from_path as read_csv_from_path, write_csv_file_to_path as write_csv_to_path, read_json_file_from_path as read_json_from_path, write_json_file_to_path as write_json_to_path

In [2]:
"""
This notebook will be used to ingest data from Yahoo Finance.

S&P 500 index is listed in the New York Stock Exchange. This means we will need to keep track of New York Time (Eastern Standard Time).

The NYSE is open from Monday through Friday 9:30 a.m. to 4:00 p.m. EST.

Only 8 days worth of 1m granularity data are allowed to be fetched per request.

The requested range must be within the last 30 days.
"""

'\nThis notebook will be used to ingest data from Yahoo Finance.\n\nS&P 500 index is listed in the New York Stock Exchange. This means we will need to keep track of New York Time (Eastern Standard Time).\n\nThe NYSE is open from Monday through Friday 9:30 a.m. to 4:00 p.m. EST.\n\nOnly 8 days worth of 1m granularity data are allowed to be fetched per request.\n\nThe requested range must be within the last 30 days.\n'

In [3]:
# FUNCTION: GET QATAR/EST HOURS OFFSET
def get_qatar_EST_hours_offset():
    # GET THE CURRENT TIME IN BOTH TIME ZONES
    now = datetime.now()
    # GET THE CURRENT TIME IN QATAR AND CURRENT TIME IN GMT
    qatar_time, est_time = qatar_tz.localize(now),  ny_tz.localize(now)
    # CALCULATE THE TIME DIFFERENCE
    time_difference = qatar_time.utcoffset() - est_time.utcoffset()
    # GET THE DIFFERENCE IN HOURS
    return time_difference.total_seconds() / 3600

# FUNCTION: GET LOCAL DATETIME NOW
def get_datetime_now_qatar():
    # RETURN DATETIME NOW
    return pd.to_datetime(qatar_tz.localize(datetime.now().replace(microsecond=0)))

# FUNCTION: GET EST DATETIME NOW
def get_datetime_now_ny():
    # COLLECT THE DATETIME NOW
    datetime_now_local = get_datetime_now_qatar()
    # RETURN QATAR TIMEZONE C
    return datetime_now_local.tz_convert(ny_tz)

# DEFINE THE QATAR TIME ZONE
qatar_tz = pytz.timezone("Asia/Qatar")
# DEFINE THE NEW YORK TIME ZONE
ny_tz = pytz.timezone("America/New_York")
# DEFINE TIMEZONE DICTIONARY
timezone_dict = {"qatar": qatar_tz, "ny": ny_tz}
# DISPLAY INFORMATION
print(f"New York is currently {get_qatar_EST_hours_offset()} hours behind Qatar (NYSE)")
# GET THE LOCAL DATE TIME (QATAR)
datetime_now_local = get_datetime_now_qatar()
# GET THE EST DATE TIME (NEW YORK)
datetime_now_ny = get_datetime_now_ny()
# DISPLAY INFORMATION
print(f"Qatar (GMT+3) Date Time:\t{datetime_now_local.tz_localize(None)}")
print(f"New York (EST) Date Time:\t{datetime_now_ny.tz_localize(None)}")

New York is currently 8.0 hours behind Qatar (NYSE)
Qatar (GMT+3) Date Time:	2025-02-18 22:24:18
New York (EST) Date Time:	2025-02-18 14:24:18


In [4]:
# DEFINE DATA SOURCE FOLDER
data_source_str = "yf"
# DEFINE DATA FOLDER
data_folder_str = f"{data_source_str}_data"
# DEFINE DATA ROOT DIRECTORY
data_root_path_str = "/Users/nico/Desktop/IMGM_data/"
# DEFINE DATA ZONES
rawzone_data_file_path_str = f"{data_root_path_str}rawzone/{data_source_str}_data/"
workzone_data_file_path_str = f"{data_root_path_str}workzone/{data_source_str}_data/"
goldzone_data_file_path_str = f"{data_root_path_str}goldzone/{data_source_str}_data/"
# DEFINE DEV DATA PATH
dev_path_str = F"{data_root_path_str}goldzone/{data_source_str}_dev/"
# DISPLAY INFORMATION
print(f"Data RawZone Storage Location:\t'{rawzone_data_file_path_str}'")
print(f"Data WorkZone Storage Location:\t'{workzone_data_file_path_str}'")
print(f"Data GoldZone Storage Location:\t'{goldzone_data_file_path_str}'")
print(f"Data GoldZone Dev Location:\t'{dev_path_str}'")

Data RawZone Storage Location:	'/Users/nico/Desktop/IMGM_data/rawzone/yf_data/'
Data WorkZone Storage Location:	'/Users/nico/Desktop/IMGM_data/workzone/yf_data/'
Data GoldZone Storage Location:	'/Users/nico/Desktop/IMGM_data/goldzone/yf_data/'
Data GoldZone Dev Location:	'/Users/nico/Desktop/IMGM_data/goldzone/yf_dev/'


In [5]:
# FUNCITON: LIST FILES IN DIRECTORY
def get_list_files_in_path(file_path_in):
    # IF THERE ARE ITEMS IN THE FILE PATH
    if len(list(os.walk(file_path_in))) > 0:
        # COLLECT ALL DATA FILE NAMES
        data_file_name_list = next(os.walk(file_path_in))[2]
        # COLLECT AND RETURN DATA FILE NAME LIST
        return [f"{file_path_in}{data_file_name}" for data_file_name in data_file_name_list]
    # IF THERE ARE NO ITEMS IN THE FILE PATH
    return []

# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_list_files_in_path(rawzone_data_file_path_str)
# workzone_data_file_path_list = get_list_files_in_path(workzone_data_file_path_str)
# goldzone_data_file_path_list = get_list_files_in_path(goldzone_data_file_path_str)
# PREVIEW LIST
rawzone_data_file_path_list

['/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241104-20241108.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241111-20241115.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241118-20241122.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241125-20241129.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241202-20241206.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241209-20241213.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241216-20241220.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241223-20241227.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20241230-20250103.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250106-20250110.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250113-20250117.csv',
 '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250120-2025012

In [6]:
# FUNCTION TO CONVERT UNIX TO DATETIME (WITH TIMEZONE)
def unix_to_datetime(unix_time_in, tz):
    # CONVERT UNIX TIMESTAMP TO DATETIME AND LOCALIZE IT TO SPECIFIED TIMEZONE
    return datetime.fromtimestamp(unix_time_in, tz)

# FUNCTION TO CONVERT DATETIME TO UNIX (WITH TIMEZONE)
def datetime_to_unix(datetime_in):
    # ENSURE THE DATETIME IS TIMEZONE-AWARE BEFORE CONVERTING TO TIMESTAMP
    if datetime_in.tzinfo is None:
        raise ValueError("Datetime object must be timezone-aware.")
    # RETURN UNIX TIME
    return datetime_in.timestamp()

# DEFINE TIMEZONE
# sample_tz = qatar_tz
sample_tz = ny_tz
# DEFINE SAMPLE DATETIME VARIABLE (Qatar timezone example)
sample_datetime = sample_tz.localize(pd.to_datetime("2019-12-30 16:00:00"))
# CONVERT DATETIME TO UNIX
converted_unix = datetime_to_unix(sample_datetime)
# CONVERT UNIX BACK TO DATETIME IN QATAR TIMEZONE
converted_datetime = unix_to_datetime(converted_unix, sample_tz)
# # DISPLAY INFORMATION
print(f"Sample Datetime:\t{sample_datetime}\tConverted To Unix:\t{converted_unix}\tBack to Datetime:\t{converted_datetime}\n")

Sample Datetime:	2019-12-30 16:00:00-05:00	Converted To Unix:	1577739600.0	Back to Datetime:	2019-12-30 16:00:00-05:00



In [7]:
# CREATE A SPARK SESSION
spark = SparkSession.builder.appName("SparkAppTest1").getOrCreate()
# PREVIEW SPARK SESSION
spark

In [8]:
# FUNCTION: QUERY CSV DATA FROM PATH
def query_rawzone_csv_yf_data_sdf(file_path_list_in, datetime1_in, datetime2_in, alert_in=True):
    # IF THE FILE PATH LIST IS NOT A LIST
    if not isinstance(file_path_list_in, list):
        # DISPLAY INFORMATION
        print(f"'{file_path_list_in}' is not a list") if alert_in==True else None
        # RETURN EMPTY VARIABLES
        return None, None
    # IF THE FILE PATH LIST IN IS EMPTY
    if (len(file_path_list_in) == 0):
        # DISPLAY INFORMATION
        print(f"No Paths Provided!") if alert_in==True else None
        # RETURN EMPTY VARIABLES
        return None, None
    # IF DATETIME 1 OR DATETIME 2 IS NOT A DATETIME OBJECT
    if (not isinstance(datetime1_in, datetime)) or (not isinstance(datetime2_in, datetime)):
        # DISPLAY INFORMATION
        print(f"'{datetime1_in}' or '{datetime2_in}' is not a datetime object") if alert_in==True else None
        # RETURN EMPTY VARIABLES
        return None, None
    # GET ALL THE PINGS DATAFRAME FOR DEVICE IDS IN LIST
    read_data_sdf = spark.read.format("csv").option("header","true").load(file_path_list_in)\
                                        .select("ny_timestamp", "open", "high", "low", "close", "volume", "stock_splits")\
                                        .withColumn("ny_timestamp", col("ny_timestamp").cast("timestamp"))\
                                        .withColumn("open", col("open").cast("Double"))\
                                        .withColumn("high", col("high").cast("Double"))\
                                        .withColumn("low", col("low").cast("Double"))\
                                        .withColumn("close", col("close").cast("Double"))\
                                        .withColumn("volume", col("volume").cast("Integer"))\
                                        .withColumn("stock_splits", col("stock_splits").cast("Double"))\
    # RETURN DATAFRAME
    return read_data_sdf

# DEFINE START DATETIME FOR DATA COLLECTION
data_start_datetime = pd.to_datetime("2024-11-04 09:30:00").tz_localize(ny_tz)
# DEFINE END DATETIME THE DATEITME NOW (NEW YORK)
data_end_datetime = get_datetime_now_ny()
# DISPLAY INFORMATION
print(f"Querying Saved Data Between ('{data_start_datetime}' - '{data_end_datetime}')")
# CALL FUNCTION TO READ DATA FROM PATH BETWEEN THE DATA START DATETIME AND DATETIME NOW (NEW YORK)
rawzone_SP500_1m_data_sdf = query_rawzone_csv_yf_data_sdf(rawzone_data_file_path_list, data_start_datetime, data_end_datetime)
# CONVERT SPARK DATAFRAME TO PANDAS DATAFRAME
rawzone_SP500_1m_data_pdf = rawzone_SP500_1m_data_sdf.toPandas()
# COLLECT THE LAST DATA DATETIME
earliest_data_datetime = rawzone_SP500_1m_data_pdf.ny_timestamp.min().tz_localize(ny_tz)
latest_data_datetime = rawzone_SP500_1m_data_pdf.ny_timestamp.max().tz_localize(ny_tz)
# DISPLAY INFORMATION
print(f"Min Timestamp: {earliest_data_datetime}")
print(f"Max Timestamp: {latest_data_datetime} (Last Data Datetime)")
# PREVIEW DATAFRAME
rawzone_SP500_1m_data_pdf

Querying Saved Data Between ('2024-11-04 09:30:00-05:00' - '2025-02-18 14:25:03-05:00')
Min Timestamp: 2024-11-04 09:30:00-05:00
Max Timestamp: 2025-01-28 13:55:00-05:00 (Last Data Datetime)


,ny_timestamp,open,high,low,close,volume,stock_splits
0,2024-11-04 09:30:00,5725.149902,5731.270020,5725.149902,5728.560059,0,0.0
1,2024-11-04 09:31:00,5728.560059,5730.399902,5724.529785,5725.419922,15064925,0.0
2,2024-11-04 09:32:00,5725.479980,5729.200195,5723.549805,5726.540039,14584166,0.0
3,2024-11-04 09:33:00,5726.569824,5726.709961,5723.129883,5723.129883,12761245,0.0
4,2024-11-04 09:34:00,5722.879883,5723.330078,5717.350098,5717.640137,12006736,0.0
...,...,...,...,...,...,...,...
17939,2025-01-28 13:51:00,6061.209961,6061.709961,6060.950195,6061.229980,3540000,0.0
17940,2025-01-28 13:52:00,6061.250000,6061.520020,6060.850098,6061.259766,3049000,0.0
17941,2025-01-28 13:53:00,6061.220215,6061.560059,6060.509766,6061.560059,3843000,0.0
17942,2025-01-28 13:54:00,6061.529785,6062.419922,6060.220215,6060.390137,4968000,0.0


In [9]:
# FUNCTION: GET ACTIVE TRADING DATE LIST
def get_active_trading_date_list(datetime1_in, datetime2_in):
    # GET LIST OF COMPLETE DATA 
    return [timestamp_object.date() for timestamp_object in pd.date_range(datetime1_in, datetime2_in, freq="24h") if timestamp_object.weekday() not in [5,6]]

# CALL FUNCTION TO GET ACTIVE TRADING DATE LIST
active_trading_date_list = get_active_trading_date_list(data_start_datetime, latest_data_datetime)
# DISPLAY INFORMATION
print(f"{len(active_trading_date_list)} Active Trading Days Between: ({data_start_datetime.date()} - {latest_data_datetime.date()})")

62 Active Trading Days Between: (2024-11-04 - 2025-01-28)


In [10]:
# FUNCTION: IDENTIFY 1 MINUTE INTERVAL STOCK TRADING TIMES
def get_active_trading_minute_list(datetime1_in, datetime2_in):
    # DEFINE DAILY START TIME AND END TIME 
    day_start_time_str, day_end_time_str = "09:30:00", "15:59:00"
    # CONVERT DAY START AND END TIME STRINGS TO TIME OBJECTS
    day_start_time_object, day_end_time_object = datetime.strptime(day_start_time_str, "%H:%M:%S").time(), datetime.strptime(day_end_time_str,"%H:%M:%S").time()
    # GET LIST OF COMPLETE MINUTE INTERVAL DATA
    complete_minute_data_list = [timestamp_object for timestamp_object in pd.date_range(datetime1_in, datetime2_in, freq="1min") if timestamp_object.date().weekday() not in [5, 6]]
    # RETURN LIST OF EXPECTED MINUTE INTERVAL DATA BY SELECTING MINUTE INTERVALS THAT FALL WITHIN TRADING HOURS
    return [timestamp_object for timestamp_object in complete_minute_data_list if (day_start_time_object <= timestamp_object.time() <= day_end_time_object)]

# CALL FUNTION TO GET ALL HISTORICAL ACTIVE TRADING MINUTE LIST
historical_trading_minute_list = get_active_trading_minute_list(data_start_datetime, latest_data_datetime)
# DISPLAY INFORMATION
print(f"{len(historical_trading_minute_list)} Historical Active Trading Minutes Between: ({data_start_datetime} - {latest_data_datetime})")

24056 Historical Active Trading Minutes Between: (2024-11-04 09:30:00-05:00 - 2025-01-28 13:55:00-05:00)


In [11]:
# FUNCTION: CREATE COMPLETE WORK WEEK TUPLES BETWEEN TWO DATE RANGES
def get_full_work_week_tuple_list(datetime1_in, datetime2_in):
    # CALL FUNCTION TO GET WORK WEEK DATES BETWEEN TWO DATETIMES
    active_trading_date_list = get_active_trading_date_list(datetime1_in, datetime2_in)
    # WEEK COUNTER
    week_counter = 0
    # CREATE A WEEK GROUP DICTIONARY
    week_group_dict = {week_counter:[]}
    # PREVIOUS WEEK DAY
    prev_weekday_ID = -1
    # ITERATE OVER DATE KEYS
    for date_object in active_trading_date_list:
        # COLLECT THE WEEK DAY ID
        weekday_ID = date_object.weekday()
        # IF THE WEEKDAY ID IS GREATER THAN PREVIOUS WEEKDAY ID
        if (prev_weekday_ID < weekday_ID):
            # SET PREVOUS WEEKDAY TO CURRENT WEEKDAY
            prev_weekday_ID = weekday_ID
        # IF PREVIOUS WEEKDAY IS IS GREATER THAN CURRENT WEEKDAY
        else:
            # RESET PREVIOUS WEEKDAY
            prev_weekday_ID = -1
            # INCREASET WEEK COUNTER
            week_counter = week_counter + 1
            # CREATE A NEW WEEK GROUP KEY VALUE ITEM
            week_group_dict[week_counter] = []
        # APPEND DATE STRING TO WEEK COUNTER
        week_group_dict[week_counter].append(date_object)
    # LIST TO HOLD WORK WEEK DATE TUPLES
    work_week_date_tuple_list = []
    # ITERATE OVER WEEK LISTS
    for week_list in week_group_dict.values():
        # COLLECT THE START AND END WEEKDAYS FOR THE WEEK
        start_weekday, end_weekday = min(week_list), max(week_list)
        # COLLECT THE START AND END WEEKDAY ID
        start_weekday_ID, end_weekday_ID = start_weekday.weekday(), end_weekday.weekday()
        # DEFINE THE EXPECTED START AND END WEEKDAY ID (MONDAY, FRIDAY)
        expected_start_weekday_ID, expected_end_weekday_ID = 0, 4
        # CALCULATE THE DIFFERECE BETWEEN THE EXPECTED AND REAL START AND END WEEKDAYS
        start_day_diff = expected_start_weekday_ID - start_weekday_ID
        end_day_diff = expected_end_weekday_ID - end_weekday_ID
        # DEFINE WORK WEEK START AND END DATE
        work_week_start_date = start_weekday if start_day_diff == 0 else start_weekday + timedelta(days = start_day_diff)
        work_week_end_date = end_weekday if end_day_diff == 0 else end_weekday + timedelta(days = end_day_diff)
        # APPEND WORK WEEK DATES TO WORK WEEK DATE TUPLE LIST
        work_week_date_tuple_list.append((work_week_start_date, work_week_end_date))
    # RETURN LIST
    return work_week_date_tuple_list

# CALL FUNCTION TO COLLECT FULL WORK WEEK TUPLE LIST BETWEEN TWO DATE RANGES
full_work_week_tuple_list = get_full_work_week_tuple_list(data_start_datetime, latest_data_datetime)
# DISPLAY INFORMATION
print(f"{len(full_work_week_tuple_list)} Active Trading Weeks Between ({data_start_datetime} - {latest_data_datetime})\n")

13 Active Trading Weeks Between (2024-11-04 09:30:00-05:00 - 2025-01-28 13:55:00-05:00)



In [12]:
# FUNCTION: DEFINE YF QUERY RULES
def validate_yf_query_dates(date1_in, date2_in, alert_in=True):
    # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
    # The requested range must be within the last 30 days.                            #
    # Only 8 days worth of 1m granularity data are allowed to be fetched per request. #
    # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
    # DEFINE QUERY START DATE LIMIT (30 DAYS PRIOR)
    query_start_date_limit = (get_datetime_now_ny() - timedelta(days=30)).date()
    # DEFINE QUERY DATE SPAN LIMIT
    date_span_limit = 8
    # GET THE DATE RANGE LIST
    date_range_list = [timestamp_object.date() for timestamp_object in pd.date_range(date1_in, date2_in)]
    # IF THE DATE RANGE LIST IS EMPTY
    if len(date_range_list) == 0:
        # DISPLAY INFORMATION
        print(f"\n\t(Alert!) Empty date range list '{date_range_list}' between {date1_in} - {date2_in}") if alert_in==True else None
        # RETURN NOTHING
        return None, None
    # DISPLAY INFORMATION
    print(f"Oldest Date:\t\t{date1_in} (Limit: {query_start_date_limit})") if alert_in==True else None
    print(f"Query Date Count Limit:\t{len(date_range_list)} (Out of {date_span_limit})") if alert_in==True else None
    # IF THE DATE COUNT IS GREATER THAN 8
    if (len(date_range_list) > date_span_limit):
        # DISPLAY INFORMATION
        print(f"\n\t(Alert!) There are too many dates '{len(date_range_list)}' between {date1_in} - {date2_in}") if alert_in==True else None
        # RETURN NOTHING
        return None, None
    # IF ANY DATE IN THE DATE SPAN IS OLDER THAN THE QUERY START DATE LIMIT
    if any(date_object <= query_start_date_limit for date_object in date_range_list):
        # DISPLAY INFORMATION
        print(f"\t(Alert!) Selected dates span before '{query_start_date_limit}' between {date1_in} - {date2_in}") if alert_in==True else None
        # RETURN NOTHING
        return None, None
    # PRINT WHITESPACE
    print()
    # RETURN 
    return date1_in, date2_in

# validate_yf_query_dates("2024-11-01", "2024-11-30")
# validate_yf_query_dates("2024-11-27", "2024-12-30")
# validate_yf_query_dates("2024-12-01", "2024-12-02")

In [13]:
# FUNCTION: DETERMINE WHETHER OR NOT TO FORM A QUERY
def generate_query_list(datetime_in):
    # GET THE CURRENT DATETIME NOW
    datetime_now_ny = get_datetime_now_ny()
    # GET ACTIVE TRADING MINUTE LIST
    active_trading_minute_list = get_active_trading_minute_list(datetime_in, datetime_now_ny)
    # IF THERE ARE ACTIVE TRADING MINUTES BETWEEN DATETIME IN AND DATETIME NOW
    if len(active_trading_minute_list) > 1:
        # DISPLAY INFORMATION
        print(f"There has been trading activity for {len(active_trading_minute_list)} minutes between {datetime_in} - {datetime_now_ny}\n")
        # IDENTIFY THE NUMBER OF WORK DAYS BETWEEN
        work_week_date_list = get_active_trading_date_list(datetime_in, datetime_now_ny)
        # DEFINE FULL WORK WEEK TUPLE LIST
        full_work_week_tuple_list = get_full_work_week_tuple_list(datetime_in, datetime_now_ny)
        # VALIDATE QUERY DATES
        validated_query_date_tuple_list = [validate_yf_query_dates(date1, date2, alert_in=True) for date1, date2 in full_work_week_tuple_list]
        # CONVERT DATES TO STRING AND RETURN
        return [(str(date1_object), str(date2_object)) for date1_object, date2_object in validated_query_date_tuple_list]
    # IF THERE HAS BEEN NO TRADING ACTIVITY
    else:
        # DISPLAY INFORMATION
        print(f"There has been no trading activity between {datetime_in} - {datetime_now_ny}\n")
        # RETURN EMPTY LIST
        return []

# CALL FUNCTION TO GENERATE DATE STRING TUPLE QUERY LIST
# date_str_tuple_query_list = generate_query_list("2024-11-30")
# date_str_tuple_query_list = generate_query_list("2024-11-01")
query_data_str_tuple_list = generate_query_list(latest_data_datetime)
# DISPLAY INFORMATION
print("Will Be Querying Between The Following Dates:")
# ITERATE OVER QUERY LIST
for i, date_str_tuple in enumerate(query_data_str_tuple_list):
    # DISPLAY INFORMATION
    print(f"Query: {i+1}\t|\t {date_str_tuple}")

There has been trading activity for 5881 minutes between 2025-01-28 13:55:00-05:00 - 2025-02-18 14:25:14-05:00

Oldest Date:		2025-01-27 (Limit: 2025-01-19)
Query Date Count Limit:	5 (Out of 8)

Oldest Date:		2025-02-03 (Limit: 2025-01-19)
Query Date Count Limit:	5 (Out of 8)

Oldest Date:		2025-02-10 (Limit: 2025-01-19)
Query Date Count Limit:	5 (Out of 8)

Oldest Date:		2025-02-17 (Limit: 2025-01-19)
Query Date Count Limit:	5 (Out of 8)

Will Be Querying Between The Following Dates:
Query: 1	|	 ('2025-01-27', '2025-01-31')
Query: 2	|	 ('2025-02-03', '2025-02-07')
Query: 3	|	 ('2025-02-10', '2025-02-14')
Query: 4	|	 ('2025-02-17', '2025-02-21')


In [33]:
"""Yahoo Finance Data Ingestion """
# AVAILABLE PERIOD FREQUENCY LIST FOR YFINANCE
yf_available_period_freq_list = ['1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y', '10y', 'ytd', 'max']
# AVAILABLE INTERVAL FREQUENCY LIST FOR YFINANCE
yf_available_interval_freq_list = ["1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h", "1d", "5d", "1wk", "1mo", "3mo"]

# # FUNCTION: QUERY DAILY INTERVAL DATA FROM YAHOO FINANCE
# def query_yf_daily_interval_data_pdf(ticker_str_in, timeframe_in, alert_in=True):
#     # DISPLAY INFORMATION
#     print(f"Querying Daily YF Data For Last '{timeframe_in}'")
#     # DEFINE THE SP_500 TICKER
#     yf_ticker = yf.Ticker(ticker_str_in)
#     # COLLECT DATAFRAME USING QUERY
#     raw_yf_SP500_24H_stock_data_pdf = yf_ticker.history(period = timeframe_in)
#     # RETURN DATAFRAME
#     return raw_yf_SP500_24H_stock_data_pdf

# FUNCTION: QUERY MINUTE INTERVAL DATA FROM YAHOO FINANCE
def query_yf_1m_interval_data_pdf(ticker_str_in, date1_in, date2_in, alert_in=True):
    # COLLECT THE CURRENT TIMESTAMP
    start_timestamp = get_datetime_now_qatar()
    # DISPLAY INFORMATION
    print(f"Attempting To Query 1m Frequency Data Between ({date1_in} - {date2_in})") if alert_in==True else None
    # DEFINE THE SP_500 TICKER
    yf_ticker = yf.Ticker(ticker_str_in)
    # DEFINE INTERVAL (1 MINUTE)
    interval_str = "1m"
    # VALIDATE QUERY DATES
    query_date1, query_date2 = validate_yf_query_dates(date1_in, date2_in, alert_in=False)
    # IF THE QUERY IS INVALID
    if (None in [query_date1, query_date2]):
        # CALL FUNCTION TO VALIDATE QUERY AGAIN
        validate_yf_query_dates(date1_in, date2_in, alert_in=True)
        # RETURN NONE
        return None
    # QUERY DATA FROM YAHOO FINANCE USING PARAMETERS
    queried_yf_1min_stock_data_pdf = yf_ticker.history(start = query_date1, end = query_date2, interval = interval_str)
    # IF THERE ARE ITEMS IN THE QUERIED DATAFRAME
    if len(queried_yf_1min_stock_data_pdf) > 0:
        # GET THE EXECUTION END TIMESTAMP
        end_timestamp = get_datetime_now_qatar()
        # GET THE TIME DIFFERENCE
        execution_seconds_duration = (end_timestamp - start_timestamp).total_seconds()
        # DISPLAY INFORMATION
        print(f"Query Returned {len(queried_yf_1min_stock_data_pdf)} Results! (in {execution_seconds_duration} seconds)\n") if alert_in==True else None
        # RETURN DATAFRAME
        return queried_yf_1min_stock_data_pdf
    # IF THERE ARE NO ITEMS IN THE QUERIED DATAFRAME
    else:
        # DISPLAY INFORMATION
        print(f"Query Did Not Return Results!\n") if alert_in==True else None
        # RETURN NOTHING
        return None

# DEFINE S&P 500 TICKER
SP500_ticker_str = "^GSPC"
# CALL FUNCTION TO COLLECT STOCK DATA FOR A PERIOD (DAYS, MONTHS YEARS) (24 HOUR INTERVALS) 
# SP500_raw_daily_interval_data_pdf = query_yf_daily_interval_data_pdf(SP500_ticker_str, "1y") # 1 YEAR
# SP500_raw_daily_interval_data_pdf = query_yf_daily_interval_data_pdf(SP500_ticker_str, "1m") # 1 MONTH
# SP500_raw_daily_interval_data_pdf = query_yf_daily_interval_data_pdf(SP500_ticker_str, "max") # MAXIMUM
# CALL FUNCTION TO COLLECT STOCK DATA FOR A PERIOD OF TIME (DEFINED INTERVAL)
# SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, "2024-11-22", "2024-12-22")
# SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, "2024-11-18", "2024-11-18")
# SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, "2024-12-16", "2024-12-20")
# SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, "2024-12-19", "2024-12-25")
# SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, '2024-12-09', '2024-12-13')
# SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, '2024-12-23', '2024-12-27')
SP500_raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(SP500_ticker_str, "2025-02-18", "2025-02-19")
# PREVIEW DATAFRAME
SP500_raw_1min_interval_data_pdf

Attempting To Query 1m Frequency Data Between (2025-02-18 - 2025-02-19)

Query Returned 298 Results! (in 1.0 seconds)



,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2025-02-18 09:30:00-05:00,6121.600098,6125.240234,6121.600098,6123.049805,0,0.0,0.0
2025-02-18 09:31:00-05:00,6123.140137,6123.740234,6118.410156,6121.129883,23113597,0.0,0.0
2025-02-18 09:32:00-05:00,6121.049805,6124.049805,6119.189941,6123.669922,16239764,0.0,0.0
2025-02-18 09:33:00-05:00,6124.129883,6125.390137,6123.209961,6123.649902,19234228,0.0,0.0
2025-02-18 09:34:00-05:00,6123.339844,6123.580078,6121.430176,6122.640137,14536455,0.0,0.0
...,...,...,...,...,...,...,...
2025-02-18 14:24:00-05:00,6103.479980,6111.040039,6103.379883,6109.910156,8776000,0.0,0.0
2025-02-18 14:25:00-05:00,6109.979980,6110.419922,6108.700195,6108.930176,4216000,0.0,0.0
2025-02-18 14:26:00-05:00,6109.029785,6109.390137,6108.549805,6108.669922,4376000,0.0,0.0


In [15]:
# FUNCTION: FORMAT YF STOCK DATA
def format_raw_yf_data_pdf(pdf_in, alert_in=True):
    # IF df_in IS NOT EMPTY
    if (isinstance(pdf_in, pd.DataFrame) == False):
        # DISPLAY INFORMATION
        print("Item Provided Is Not A DataFrame") if alert_in == True else None
        # EXIT FUNCTION
        return None
    # IF pdf_in IS EMPTY
    if (len(pdf_in) == 0):
        # DISPLAY INFORMATION
        print("DataFrame Provided Is Empty! Proceeding...") if alert_in == True else None
        # EXIT FUNCTION
        return None
    # COPY THE DATAFRAME
    stock_data_pdf = pdf_in.copy()
    # DEFINE TIMESTAMP COLUMN NAME
    time_column_name_str = "ny_timestamp"
    # IF TIMESTAMP IS NOT IN THE DATAFRAME
    if "timestamp" not in stock_data_pdf.columns:
        # RESET INDEX TO CONVERT DATE INDEX TO A COLUMN
        stock_data_pdf = stock_data_pdf.reset_index()
        # LOWERCASE ALL THE COLUMN NAMES
        stock_data_pdf.columns = [i.lower().replace(" ","_") for i in stock_data_pdf.columns]
        # RENAME DATE OR DATETIME TO TIMESTAMP AND RETURN DATAFRAME 
        stock_data_pdf = stock_data_pdf.rename(columns={"date":time_column_name_str,"datetime":time_column_name_str})
        # REMOVE TIMEZONE AWARENESS FROM TIMESTAMP
        stock_data_pdf[time_column_name_str] = stock_data_pdf[time_column_name_str].apply(lambda x: x.replace(tzinfo=None))
        # REMOVE DIVIDENDS FROM DATAFRAME
        stock_data_pdf = stock_data_pdf.drop(["dividends"], axis=1)
    # CONVERT TIMESTAMP TO PANDAS DATETIME
    stock_data_pdf.ny_timestamp = pd.to_datetime(stock_data_pdf.ny_timestamp)
    # COLLECT GENERAL STATISTIC VARIABLES
    min_timestamp = stock_data_pdf.ny_timestamp.min()
    max_timestamp = stock_data_pdf.ny_timestamp.max()
    row_count = len(stock_data_pdf)
    # DISPLAY INFORMATION
    print(f"Minimum Data Timestamp: {min_timestamp}") if alert_in == True else None
    print(f"Maximum Data Timestamp: {max_timestamp}") if alert_in == True else None
    print(f"Data Row Count: {row_count}")
    print(f"Current timestamp (ny): {get_datetime_now_ny()}")
    # RETURN DATAFRAME
    return stock_data_pdf

# # CALL FUNCTION TO FORMAT YF STOCK DATA
# raw_SP500_1min_data_pdf = format_raw_yf_data_pdf(SP500_raw_1min_interval_data_pdf)
# # PREVIEW DATAFRAME
# raw_SP500_1min_data_pdf

In [16]:
# FUNCTION: COLLECT EXISTING DATA LIST
def get_existing_week_data_tuple_list(data_file_path_list_in):
    # DEFINE DATA FILE NAME PREFIX
    data_file_name_prefix_str = "yf_SP500_1m_"
    # COMPLETE DEFINE DATA FILE PATH AND PREFIX
    complete_data_file_name_prefix = f"{rawzone_data_file_path_str}{data_file_name_prefix_str}"
    # DEFINE DATA FILE NAME EXTENSION
    data_file_extention = ".csv"
    # EXTRATCT THE FILE NAME FROM THE PATH
    data_file_date_range_str_list = [data_file_path.replace(complete_data_file_name_prefix,"").replace(data_file_extention,"") for data_file_path in data_file_path_list_in]
    # RETURN THE SPLIT DATA FILE DATE LIST
    return [tuple(data_file_date_range_str.split("-")) for data_file_date_range_str in data_file_date_range_str_list]

# COLLECT ALL EXISTING WEEK DATA TUPLE
existing_data_str_tuple_list = get_existing_week_data_tuple_list(rawzone_data_file_path_list)
# PREVIEW LIST
existing_data_str_tuple_list

[('20241104', '20241108'),
 ('20241111', '20241115'),
 ('20241118', '20241122'),
 ('20241125', '20241129'),
 ('20241202', '20241206'),
 ('20241209', '20241213'),
 ('20241216', '20241220'),
 ('20241223', '20241227'),
 ('20241230', '20250103'),
 ('20250106', '20250110'),
 ('20250113', '20250117'),
 ('20250120', '20250124'),
 ('20250127', '20250131')]

In [17]:
# FUNCTION: CALL MINUTE INTERVAL DATA USING 
def multi_query_yf_minute_interval_data_pdf(ticker_str_in, query_data_str_tuple_list_in, existing_data_str_tuple_list_in):
    # IDENTIFY ALL DATA STR TUPLE THAT HAVE NOT BEEN QUERIED
    required_data_str_tuple_list = [data_str_tuple for data_str_tuple in query_data_str_tuple_list_in if data_str_tuple not in existing_data_str_tuple_list_in]
    # DISPLAY INFORMATION
    print(f"There are {len(existing_data_str_tuple_list_in)} date ranges (weeks) that have already been queried")
    print(f"There are {len(query_data_str_tuple_list_in)} date ranges (weeks) that need to be queried")
    print(f"There are {len(required_data_str_tuple_list)} date ranges (weeks) that need to be queried")
    # DATAFRAME LIST
    pdf_list = []
    # ITERATE OVER DATE TUPLE LIST IN
    for date1, date2 in required_data_str_tuple_list:
        # CALL FUNCTION TO COLLECT STOCK DATA FOR A PERIOD OF TIME (DEFINED INTERVAL)
        raw_1min_interval_data_pdf = query_yf_1m_interval_data_pdf(ticker_str_in, str(date1), str(date2), alert_in=True)
        # CALL FUNCTION TO FORMAT YF STOCK DATA
        clean_1min_interval_data_pdf = format_raw_yf_data_pdf(raw_1min_interval_data_pdf, alert_in=False)
        # IF ITEM IS A DATAFRAME
        if isinstance(clean_1min_interval_data_pdf, pd.DataFrame):
            # APPEND DATAFRAME TO PDF LIST
            pdf_list.append(clean_1min_interval_data_pdf)
    # DEFINE COLUMN NAME LIST
    column_name_list = ["ny_timestamp", "open", "high", "low", "close", "volume", "stock_splits"]
    # RETURN COMBINED DATAFRAMES
    return pd.concat(pdf_list, axis=0) if len(pdf_list) > 0 else pd.DataFrame(columns = column_name_list)

# CALL FUNCTION TO QUERY MULTIPLE DATE RANGES OF YF MINUTE INTERVAL DATA
multi_query_SP500_1m_data_pdf = multi_query_yf_minute_interval_data_pdf(SP500_ticker_str, query_data_str_tuple_list, existing_data_str_tuple_list)
# DISPLAY INFORMATION
print(f"Min Timestamp: {multi_query_SP500_1m_data_pdf.ny_timestamp.min()}")
print(f"Max Timestamp: {multi_query_SP500_1m_data_pdf.ny_timestamp.max()}")
# PREVIEW DATAFRAME
multi_query_SP500_1m_data_pdf

There are 13 date ranges (weeks) that have already been queried
There are 4 date ranges (weeks) that need to be queried
There are 4 date ranges (weeks) that need to be queried
Attempting To Query 1m Frequency Data Between (2025-01-27 - 2025-01-31)

Query Returned 1554 Results! (in 4.0 seconds)

Data Row Count: 1554
Current timestamp (ny): 2025-02-18 14:25:18-05:00
Attempting To Query 1m Frequency Data Between (2025-02-03 - 2025-02-07)

Query Returned 1555 Results! (in 0.0 seconds)

Data Row Count: 1555
Current timestamp (ny): 2025-02-18 14:25:18-05:00
Attempting To Query 1m Frequency Data Between (2025-02-10 - 2025-02-14)

Query Returned 1551 Results! (in 0.0 seconds)

Data Row Count: 1551
Current timestamp (ny): 2025-02-18 14:25:18-05:00
Attempting To Query 1m Frequency Data Between (2025-02-17 - 2025-02-21)

Query Returned 295 Results! (in 1.0 seconds)

Data Row Count: 295
Current timestamp (ny): 2025-02-18 14:25:19-05:00
Min Timestamp: 2025-01-27 09:30:00
Max Timestamp: 2025-02-18 1

,ny_timestamp,open,high,low,close,volume,stock_splits
0,2025-01-27 09:30:00,5969.040039,5969.040039,5962.919922,5966.859863,0,0.0
1,2025-01-27 09:31:00,5984.189941,5990.939941,5983.620117,5990.569824,54966175,0.0
2,2025-01-27 09:32:00,5990.359863,5993.149902,5989.740234,5990.750000,23623355,0.0
3,2025-01-27 09:33:00,5990.350098,5995.379883,5987.620117,5993.209961,21954940,0.0
4,2025-01-27 09:34:00,5992.939941,5994.910156,5988.750000,5992.319824,18830357,0.0
...,...,...,...,...,...,...,...
290,2025-02-18 14:21:00,6100.470215,6102.859863,6100.470215,6102.560059,4543000,0.0
291,2025-02-18 14:22:00,6102.540039,6103.759766,6102.450195,6103.759766,4897000,0.0
292,2025-02-18 14:23:00,6103.649902,6103.649902,6102.950195,6103.520020,4194000,0.0
293,2025-02-18 14:24:00,6103.479980,6111.040039,6103.379883,6109.910156,8776000,0.0


In [18]:
# FUNCTION: PARTITION DATA BY DATE
def get_work_week_partition_pdf_dict(pdf_in, time_column_in):
    # IF THE DATAFRAME IS NOT A DATAFRAME
    if isinstance(pdf_in, pd.DataFrame) == False:
        # DISPLAY INFORMATION
        print(f"Item passed is not a DataFrame!\nProceeding...")
        # RETURN EMPTY DICTIONARY
        return {}
    # IF DATAFRAME IN IS EMPTY
    if len(pdf_in) == 0:
        # DISPLAY INFORMATION
        print("DataFrame is empty!\nProceeding...")
        # RETURN EMPTY DICTIONARY
        return {}
    # COLLECT COLUMN NAME LIST    
    column_name_list = pdf_in.columns.tolist()
    # IF THE TIME COLUMN IS NOT IN THE DATAFRAME
    if time_column_in not in column_name_list:
        # DISPLAY INFORMATION
        print(f"'{time_column_in}' is not in {column_name_list}")
        # RETURN EMPTY DICTIONARY
        return {}
    # COPY DATAFRAME
    pdf_out = pdf_in.copy()
    # IF THE TIME COLUMN IN CONTAINS UNIX
    if "unix" in time_column_in:
        # COLLECT THE LOCATION FROM time_column_in
        location_str = time_column_in.split("_")[0]
        # COLLECT THE TIMEZONE BASED ON THE LOCATION
        tz = timezone_dict[location_str]
        # SET THE TIME COLUMN IN TO THE NEW TIMESTAMP COLUMN NAME
        new_time_column = f"{location_str}_timestamp"
        # CREATE A NEW COLUMN WITH THE TIMESTAMP (AND REMOVE THE TIMEZONE)
        pdf_out[new_time_column] = pdf_out[time_column_in].apply(lambda x: unix_to_datetime(x, tz)).dt.tz_localize(None)
        # SET THE time_column_in TO THE time_column
        time_column_in = new_time_column
    # CONVERT TIMESTAMP COLUMN TO PANDAS DATETIME
    pdf_out[time_column_in] = pd.to_datetime(pdf_out[time_column_in])
    # CREATE A DATE COLUMN  
    pdf_out["date"] = pdf_out[time_column_in].apply(lambda x: x.date())
    # GROUP DATAFRAME BY DATE
    date_group = pdf_out.groupby("date")
    # GET DATE KEYS
    date_keys = list(date_group.groups)
    # COLLECT THE MINIMUM AND MAXIMUM DATE
    min_date_object, max_date_object = min(date_keys), max(date_keys)
    # CALL FUNCTION TO COLLECT FULL WORK WEEK TUPLE LIST BETWEEN TWO DATE RANGES
    full_work_week_tuple_list = get_full_work_week_tuple_list(min_date_object, max_date_object)
    # LIST OF DATAFRAMES
    work_week_date_pdf_dict = {}
    # LIST OF DATAFRAMES
    for date1_object, date2_object in full_work_week_tuple_list:
        # DEFINE DATA START TIMESTAMP
        start_timestamp = pd.to_datetime(str(date1_object) + " 09:30:00")
        # DEFINE DATA END TIMESTAMP
        end_timestamp = pd.to_datetime(str(date2_object) + " 15:59:00")
        # DEFINE DICTIONARY KEY
        week_range_key_str = f"{date1_object.strftime('%Y%m%d')}-{date2_object.strftime('%Y%m%d')}"
        # POPULATE DICTIONARY
        work_week_date_pdf_dict[week_range_key_str] = pdf_out[pdf_out[time_column_in].between(start_timestamp, end_timestamp)][column_name_list].reset_index(drop=True)
    # RETURN DICTIONARY
    return work_week_date_pdf_dict

# GET DATE PARTITION DATAFRAME LIST
rawzone_week_str_pdf_dict = get_work_week_partition_pdf_dict(multi_query_SP500_1m_data_pdf, "ny_timestamp")
# ITERATE OVER DICTIONARY
for week_str, pdf in rawzone_week_str_pdf_dict.items():
    # DISPLAY INFORMATION
    print(f"Week: '{week_str},'\tDataFrame Row Count: {len(pdf)},\t{pdf.columns.tolist()}")

Week: '20250127-20250131,'	DataFrame Row Count: 1554,	['ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20250203-20250207,'	DataFrame Row Count: 1555,	['ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20250210-20250214,'	DataFrame Row Count: 1551,	['ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20250217-20250221,'	DataFrame Row Count: 295,	['ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']


In [19]:
# FUNCTION: SAVE DATA TO RAWZONE 
def save_pdf_dict_to_path(data_dict_in, file_path_in):
    # IF DICTIONARY IS EMPTY
    if len(data_dict_in) == 0:
        # DISPLAY INFORMATION
        print("Data dictionary is empty!\nProceeding...")
    # DEFINE DATA FILE NAME PREFIX
    data_file_name_prefix_str = f"{data_source_str}_SP500_1m_"
    # DEFINE DATA FILE EXTENSION
    data_file_extension_str = ".csv"
    # ITERATE OVER DICTIONARY
    for file_name_date_str, write_pdf in data_dict_in.items():
        # DEFINE FILE PATH STRING
        data_file_path_str = f"{file_path_in}{data_file_name_prefix_str}{file_name_date_str}{data_file_extension_str}"
        # READ THE CURRENT DATA FILE AS STRING VALUES
        read_pdf = read_csv_from_path(data_file_path_str, alert_in=False)
        # IF THE READ DATAFRAME EXISTS
        if isinstance(read_pdf, pd.DataFrame):
            # DISPLAY INFORMATION
            print(f"Existing DataFrame Found For '{data_file_name_prefix_str}{file_name_date_str}'")
            # CONVERT VALUES TO STRINGS
            str_read_pdf = read_pdf.astype(str)
            # CONVERT THE WRITE DATAFRAME VALUES TO STRING
            str_write_pdf = write_pdf.astype(str)
            # IF THE READ DATAFRAME IS EQUAL TO THE WRITE DATAFRAME
            if str_read_pdf.equals(str_write_pdf) == False:
                # DISPLAY INFORMATION
                print(f"Queried DataFrame Is Not Equal To Existing DataFrame")
                # IF THE QUERIED DATAFRAME HAS LESS ROWS THAN THE EXISTING DATAFRAME
                if len(str_write_pdf) < len(str_read_pdf):
                    # RAISE EXCEPTION
                    raise Exception(f"Queried DataFrame For {file_name_date_str} Contains Less Rows Than Existing DataFrame\nInvestigate!!!")
                # IF THE QUERIED DATAFRAME HAS MORE ROWS THAN THE EXISTING DATAFRAME
                else:
                    # DISPLAY INFORMATION
                    print(f"Queried DataFrame (row count: {len(write_pdf)}) Contains More Rows Than Existing DataFrame (row_count: {len(read_pdf)})\nOverwriting...")
                    # OVERWRITE DATAFRAME
                    write_csv_to_path(write_pdf, data_file_path_str, "W")
            # IF READ DATAFRAME IS EQUAL TO WRITE DATAFRAME
            else:
                # DISPLAY INFORMATION
                print(f"Queried DataFrame is Equal To Existing DataFrame\nProceeding...")
        # IF THE DATAFRAME DOES NOT ALREADY EXIST
        else:
            # WRITE DATAFRAME
            write_csv_to_path(write_pdf, data_file_path_str, "W")
    
# CALL FUNCTION TO SAVE DATA DICTIONARY TO RAWZONE
save_pdf_dict_to_path(rawzone_week_str_pdf_dict, rawzone_data_file_path_str)

Existing DataFrame Found For 'yf_SP500_1m_20250127-20250131'
Queried DataFrame Is Not Equal To Existing DataFrame
Queried DataFrame (row count: 1554) Contains More Rows Than Existing DataFrame (row_count: 654)
Overwriting...
Saving In Overwrite Mode
File '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250127-20250131.csv' Found!
File: '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250127-20250131.csv' Has Been Deleted!
File: '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250127-20250131.csv' Has Been Saved!
Saving In Overwrite Mode
File '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250203-20250207.csv' Not Found!
File: '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250203-20250207.csv' Has Been Saved!
Saving In Overwrite Mode
File '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250210-20250214.csv' Not Found!
File: '/Users/nico/Desktop/IMGM_data/rawzone/yf_data/yf_SP500_1m_20250210-20250214.csv' Has Been

In [20]:
# GET UPDATED RAWZONE DATA FILE PATH LIST
updated_rawzone_data_file_path_list = get_list_files_in_path(rawzone_data_file_path_str)
# CALL FUNCTION TO READ DATA FROM PATH BETWEEN THE DATA START DATETIME AND DATETIME NOW (NEW YORK)
reloaded_rawzone_SP500_1m_data_sdf = query_rawzone_csv_yf_data_sdf(updated_rawzone_data_file_path_list, data_start_datetime, data_end_datetime)
# CONVERT SPARK DATAFRAME TO PANDAS DATAFRAME
reloaded_rawzone_SP500_1m_data_pdf = reloaded_rawzone_SP500_1m_data_sdf.toPandas()
# COLLECT PREVIOUS DATA ROW COUNT
prev_row_count = len(rawzone_SP500_1m_data_pdf)
# COLLET UPDATED DATA ROW COUNT
updated_row_count = len(reloaded_rawzone_SP500_1m_data_pdf)
# CALCULATE TEH ROW COUNT DIFFERENCE
row_count_diff = updated_row_count - prev_row_count
# DEFINE DESCRIPTION STRING
description_str = "Data has not been updated!" if (row_count_diff == 0) else "Data has been updated!"
# DISPLAY INFORMATION
print(f"Previous Complete Data Row Count: {prev_row_count}")
print(f"Updated Complete Data Row Count: {updated_row_count} (increase of {row_count_diff} rows)")
print(description_str)
# PREVIEW DATAFRAME
reloaded_rawzone_SP500_1m_data_pdf

Previous Complete Data Row Count: 17944
Updated Complete Data Row Count: 22245 (increase of 4301 rows)
Data has been updated!


,ny_timestamp,open,high,low,close,volume,stock_splits
0,2024-11-04 09:30:00,5725.149902,5731.270020,5725.149902,5728.560059,0,0.0
1,2024-11-04 09:31:00,5728.560059,5730.399902,5724.529785,5725.419922,15064925,0.0
2,2024-11-04 09:32:00,5725.479980,5729.200195,5723.549805,5726.540039,14584166,0.0
3,2024-11-04 09:33:00,5726.569824,5726.709961,5723.129883,5723.129883,12761245,0.0
4,2024-11-04 09:34:00,5722.879883,5723.330078,5717.350098,5717.640137,12006736,0.0
...,...,...,...,...,...,...,...
22240,2025-02-18 14:21:00,6100.470215,6102.859863,6100.470215,6102.560059,4543000,0.0
22241,2025-02-18 14:22:00,6102.540039,6103.759766,6102.450195,6103.759766,4897000,0.0
22242,2025-02-18 14:23:00,6103.649902,6103.649902,6102.950195,6103.520020,4194000,0.0
22243,2025-02-18 14:24:00,6103.479980,6111.040039,6103.379883,6109.910156,8776000,0.0


In [21]:
# FUNCTION: CONVERT TIMEZONE BETWEEN QATAR TO NEW YORK
def get_qatar_ny_timestamp(pdf_in, time_column_str_in, current_timezone_str_in):
    # CHECK IF TIMEZONE IN IS QATAR OR NEW YORK
    assert current_timezone_str_in in ["ny", "qatar"], "Please enter either 'ny' or 'qatar'"
    # IF THE DATAFRAME IS NOT A DATAFRAME
    if isinstance(pdf_in, pd.DataFrame) == False:
        # DISPLAY INFORMATION
        print(f"Item passed is not a DataFrame!\nProceeding...")
        # RETURN NOTHING
        return None
    # IF THE DATAFRAME IS NOT A DATAFRAME
    if isinstance(pdf_in, pd.DataFrame) == False:
        # DISPLAY INFORMATION
        print(f"DataFrame passed is empty!\nProceeding...")
        # RETURN NOTHING 
        return None
    # COLUMN NAME LIST
    column_name_list = pdf_in.columns.tolist()
    # IF THE TIME COLUMN IN IS NOT IN THE COLUNS
    if time_column_str_in not in column_name_list:
        # DISPLAY INFORMATION
        print(f"'{time_column_str_in}' Is Not In DataFrame Columns!\nProceeding...")
        # RETURN NOTHING 
        return None
    # COPY THE DATAFRAME
    pdf_out = pdf_in.copy()
    # REMOVE ANY TIMESTAMP COLUMNS FROM THE DATAFRAME
    non_timestamp_column_name_list = [column_name for column_name in column_name_list if "timestamp" not in column_name]
    # DEFINE THE QATAR TIME ZONE
    qatar_tz = pytz.timezone("Asia/Qatar")
    # DEFINE THE NEW YORK TIME ZONE
    new_york_tz = pytz.timezone("America/New_York")
    # INPUT TIMEZONE DICTIONARY
    input_timezone_dict = {
                        # IF THE INPUT IS QATAR, GET THE NEW YORK TIMEZONE DATA
                        "qatar": [qatar_tz, new_york_tz, "qatar", "ny"],
                        # IF THE INPUT IS NEW YORK, GET THE QATAR TIMEZONE DATA
                        "ny":    [new_york_tz, qatar_tz, "ny", "qatar"]
                        }
    # COLLECT THE INPUT TIMEZONE, OUTPUT TIMEZONE, INPUT TIMEZONE STRING, OUTPUT TIMEZONE STRING
    input_tz, output_tz, input_timezone_str, output_timezone_str = input_timezone_dict[current_timezone_str_in]
    # DEFINE INPUT COLUMN NAME
    new_input_column_name_str, new_output_column_name_str = f"{input_timezone_str}_timestamp", f"{output_timezone_str}_timestamp"
    # CONVERT LOCALIZE TIMESTAMP TO QATAR TIMEZONE
    pdf_out[new_input_column_name_str] = pdf_out[time_column_str_in].dt.tz_localize(input_tz)
    # CONVERT THE LOCALIZED TIMESTAMP TO NEW YORK TIMEZONE
    pdf_out[new_output_column_name_str] = pdf_out[new_input_column_name_str].dt.tz_convert(output_tz).dt.tz_localize(None)
    # REMOVE THE LOCALIZATION OF QATAR TIMESTAMP
    pdf_out[new_input_column_name_str] = pdf_out[new_input_column_name_str].dt.tz_localize(None)
    # RETURN DATAFRAME
    return pdf_out[["qatar_timestamp", "ny_timestamp"] + non_timestamp_column_name_list]

# CALL FUNCTION TO GET THE QATAR TIMESTAMP USING THE NEW YORK TIMESTAMP
rawzone_SP500_1m_data_qatar_ny_pdf = get_qatar_ny_timestamp(reloaded_rawzone_SP500_1m_data_pdf, "ny_timestamp", "ny")
# PREVIEW DATAFRAME
rawzone_SP500_1m_data_qatar_ny_pdf

,qatar_timestamp,ny_timestamp,open,high,low,close,volume,stock_splits
0,2024-11-04 17:30:00,2024-11-04 09:30:00,5725.149902,5731.270020,5725.149902,5728.560059,0,0.0
1,2024-11-04 17:31:00,2024-11-04 09:31:00,5728.560059,5730.399902,5724.529785,5725.419922,15064925,0.0
2,2024-11-04 17:32:00,2024-11-04 09:32:00,5725.479980,5729.200195,5723.549805,5726.540039,14584166,0.0
3,2024-11-04 17:33:00,2024-11-04 09:33:00,5726.569824,5726.709961,5723.129883,5723.129883,12761245,0.0
4,2024-11-04 17:34:00,2024-11-04 09:34:00,5722.879883,5723.330078,5717.350098,5717.640137,12006736,0.0
...,...,...,...,...,...,...,...,...
22240,2025-02-18 22:21:00,2025-02-18 14:21:00,6100.470215,6102.859863,6100.470215,6102.560059,4543000,0.0
22241,2025-02-18 22:22:00,2025-02-18 14:22:00,6102.540039,6103.759766,6102.450195,6103.759766,4897000,0.0
22242,2025-02-18 22:23:00,2025-02-18 14:23:00,6103.649902,6103.649902,6102.950195,6103.520020,4194000,0.0
22243,2025-02-18 22:24:00,2025-02-18 14:24:00,6103.479980,6111.040039,6103.379883,6109.910156,8776000,0.0


In [22]:
# FUNCTION: FORMAT DATA FOR WORKZONE
def format_yf_pdf_for_workzone(pdf_in):
    # COPY THE DATAFRAME
    pdf_out = pdf_in.copy()
    # ADD QATAR AND NEW YORK TIME TO DATAFRAME
    pdf_out = get_qatar_ny_timestamp(pdf_out, "ny_timestamp", "ny")
    # DEFINE COLUMN NAME LIST
    column_name_list = ["qatar_timestamp", "ny_timestamp", "open", "high", "low", "close", "volume", "stock_splits"]
    # RETURN DATAFRAME
    return pdf_out[column_name_list]

# CALL FUNCTION TO FORMAT THE YF DATA FOR WORKZONE
work_SP500_1m_data_pdf = format_yf_pdf_for_workzone(reloaded_rawzone_SP500_1m_data_pdf)
# DISPLAY INFORMATION
print(f"Min Timestamp: {work_SP500_1m_data_pdf.ny_timestamp.min()}")
print(f"Max Timestamp: {work_SP500_1m_data_pdf.ny_timestamp.max()}")
# PREVIEW DATAFRAME
work_SP500_1m_data_pdf

Min Timestamp: 2024-11-04 09:30:00
Max Timestamp: 2025-02-18 14:25:00


,qatar_timestamp,ny_timestamp,open,high,low,close,volume,stock_splits
0,2024-11-04 17:30:00,2024-11-04 09:30:00,5725.149902,5731.270020,5725.149902,5728.560059,0,0.0
1,2024-11-04 17:31:00,2024-11-04 09:31:00,5728.560059,5730.399902,5724.529785,5725.419922,15064925,0.0
2,2024-11-04 17:32:00,2024-11-04 09:32:00,5725.479980,5729.200195,5723.549805,5726.540039,14584166,0.0
3,2024-11-04 17:33:00,2024-11-04 09:33:00,5726.569824,5726.709961,5723.129883,5723.129883,12761245,0.0
4,2024-11-04 17:34:00,2024-11-04 09:34:00,5722.879883,5723.330078,5717.350098,5717.640137,12006736,0.0
...,...,...,...,...,...,...,...,...
22240,2025-02-18 22:21:00,2025-02-18 14:21:00,6100.470215,6102.859863,6100.470215,6102.560059,4543000,0.0
22241,2025-02-18 22:22:00,2025-02-18 14:22:00,6102.540039,6103.759766,6102.450195,6103.759766,4897000,0.0
22242,2025-02-18 22:23:00,2025-02-18 14:23:00,6103.649902,6103.649902,6102.950195,6103.520020,4194000,0.0
22243,2025-02-18 22:24:00,2025-02-18 14:24:00,6103.479980,6111.040039,6103.379883,6109.910156,8776000,0.0


In [23]:
# GET DATE PARTITION DATAFRAME LIST
workzone_week_str_pdf_dict = get_work_week_partition_pdf_dict(work_SP500_1m_data_pdf, "ny_timestamp")
# ITERATE OVER DICTIONARY
for week_str, pdf in workzone_week_str_pdf_dict.items():
    # DISPLAY INFORMATION
    print(f"Week: '{week_str},'\tDataFrame Row Count: {len(pdf)},\t{pdf.columns.tolist()}")

Week: '20241104-20241108,'	DataFrame Row Count: 1947,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20241111-20241115,'	DataFrame Row Count: 1910,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20241118-20241122,'	DataFrame Row Count: 1552,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20241125-20241129,'	DataFrame Row Count: 1168,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20241202-20241206,'	DataFrame Row Count: 1558,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20241209-20241213,'	DataFrame Row Count: 1555,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low', 'close', 'volume', 'stock_splits']
Week: '20241216-20241220,'	DataFrame Row Count: 1555,	['qatar_timestamp', 'ny_timestamp', 'open', 'high', 'low',

In [24]:
# CALL FUNCTION TO SAVE DATA DICTIONARY TO WORKZONE
save_pdf_dict_to_path(workzone_week_str_pdf_dict, workzone_data_file_path_str)

Existing DataFrame Found For 'yf_SP500_1m_20241104-20241108'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241111-20241115'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241118-20241122'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241125-20241129'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241202-20241206'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241209-20241213'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241216-20241220'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing DataFrame Found For 'yf_SP500_1m_20241223-20241227'
Queried DataFrame is Equal To Existing DataFrame
Proceeding...
Existing

In [25]:
# CALL FUNCTION TO GET ALL FILES IN PATH
workzone_data_file_path_list = get_list_files_in_path(workzone_data_file_path_str)
# PREVIEW LIST
workzone_data_file_path_list

['/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241104-20241108.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241111-20241115.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241118-20241122.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241125-20241129.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241202-20241206.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241209-20241213.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241216-20241220.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241223-20241227.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20241230-20250103.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20250106-20250110.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_20250113-20250117.csv',
 '/Users/nico/Desktop/IMGM_data/workzone/yf_data/yf_SP500_1m_2025

In [26]:
# FUNCTION: QUERY CSV DATA FROM PATH
def query_workzone_csv_yf_data_sdf(file_path_list_in, datetime1_in, datetime2_in, alert_in=True):
    # IF THE FILE PATH LIST IS NOT A LIST
    if not isinstance(file_path_list_in, list):
        # DISPLAY INFORMATION
        print(f"'{file_path_list_in}' is not a list") if alert_in==True else None
        # RETURN EMPTY VARIABLES
        return None, None
    # IF THE FILE PATH LIST IN IS EMPTY
    if (len(file_path_list_in) == 0):
        # DISPLAY INFORMATION
        print(f"No Paths Provided!") if alert_in==True else None
        # RETURN EMPTY VARIABLES
        return None, None
    # IF DATETIME 1 OR DATETIME 2 IS NOT A DATETIME OBJECT
    if (not isinstance(datetime1_in, datetime)) or (not isinstance(datetime2_in, datetime)):
        # DISPLAY INFORMATION
        print(f"'{datetime1_in}' or '{datetime2_in}' is not a datetime object") if alert_in==True else None
        # RETURN EMPTY VARIABLES
        return None, None
    # GET ALL THE PINGS DATAFRAME FOR DEVICE IDS IN LIST
    read_data_sdf = spark.read.format("csv").option("header","true").load(file_path_list_in)\
                                        .withColumn("qatar_timestamp", col("qatar_timestamp").cast("timestamp"))\
                                        .withColumn("ny_timestamp", col("ny_timestamp").cast("timestamp"))\
                                        .withColumn("open", col("open").cast("Double"))\
                                        .withColumn("high", col("high").cast("Double"))\
                                        .withColumn("low", col("low").cast("Double"))\
                                        .withColumn("close", col("close").cast("Double"))\
                                        .withColumn("volume", col("volume").cast("Integer"))\
                                        .withColumn("stock_splits", col("stock_splits").cast("Double"))\
                                        .sort("ny_timestamp")
    # RETURN DATAFRAME
    return read_data_sdf

# READ WORKZONE DATA
workzone_SP500_1m_data_sdf = query_workzone_csv_yf_data_sdf(workzone_data_file_path_list, data_start_datetime, data_end_datetime)
# CONVERT DATAFRAME FROM SPARK TO PANDAS
workzone_SP500_1m_data_pdf = workzone_SP500_1m_data_sdf.toPandas()
# PREVIEW DATAFRAME
workzone_SP500_1m_data_pdf

,qatar_timestamp,ny_timestamp,open,high,low,close,volume,stock_splits
0,2024-11-04 17:30:00,2024-11-04 09:30:00,5725.149902,5731.270020,5725.149902,5728.560059,0,0.0
1,2024-11-04 17:31:00,2024-11-04 09:31:00,5728.560059,5730.399902,5724.529785,5725.419922,15064925,0.0
2,2024-11-04 17:32:00,2024-11-04 09:32:00,5725.479980,5729.200195,5723.549805,5726.540039,14584166,0.0
3,2024-11-04 17:33:00,2024-11-04 09:33:00,5726.569824,5726.709961,5723.129883,5723.129883,12761245,0.0
4,2024-11-04 17:34:00,2024-11-04 09:34:00,5722.879883,5723.330078,5717.350098,5717.640137,12006736,0.0
...,...,...,...,...,...,...,...,...
22240,2025-02-18 22:21:00,2025-02-18 14:21:00,6100.470215,6102.859863,6100.470215,6102.560059,4543000,0.0
22241,2025-02-18 22:22:00,2025-02-18 14:22:00,6102.540039,6103.759766,6102.450195,6103.759766,4897000,0.0
22242,2025-02-18 22:23:00,2025-02-18 14:23:00,6103.649902,6103.649902,6102.950195,6103.520020,4194000,0.0
22243,2025-02-18 22:24:00,2025-02-18 14:24:00,6103.479980,6111.040039,6103.379883,6109.910156,8776000,0.0


In [27]:
# ADD A DATE COLUMN
workzone_SP500_1m_data_pdf["ny_date"] = workzone_SP500_1m_data_pdf.ny_timestamp.apply(lambda x: x.date())
# AGGREGATE DATE DATA BY MINIMUM AND MAXIMUM TIMESTAMP
workzone_SP500_1m_data_pdf.groupby(["ny_date"]).agg(min_timestamp=("ny_timestamp", "min"), max_timestamp=("ny_timestamp","max"), row_count=("ny_timestamp","count")).reset_index()

,ny_date,min_timestamp,max_timestamp,row_count
0,2024-11-04,2024-11-04 09:30:00,2024-11-04 15:59:00,390
1,2024-11-05,2024-11-05 09:30:00,2024-11-05 15:59:00,390
2,2024-11-06,2024-11-06 09:30:00,2024-11-06 15:59:00,389
3,2024-11-07,2024-11-07 09:30:00,2024-11-07 15:59:00,390
4,2024-11-08,2024-11-08 09:30:00,2024-11-08 15:59:00,388
5,2024-11-11,2024-11-11 09:30:00,2024-11-11 15:59:00,380
6,2024-11-12,2024-11-12 09:30:00,2024-11-12 15:59:00,375
7,2024-11-13,2024-11-13 09:30:00,2024-11-13 15:59:00,382
8,2024-11-14,2024-11-14 09:30:00,2024-11-14 15:59:00,383
9,2024-11-15,2024-11-15 09:30:00,2024-11-15 15:59:00,390
